##OUR NOTES:
**Progress & Keyword Extraction**

**Lyn:** radiology

**Stephen:** oncology

**Leo:** pathology

# Code below divided into 4 sections:
#### Fetcher, Classifier, Storage & Agent


Install any missing libraries

In [ ]:
!pip -q install \
  pymupdf \
  groq \
  python-dotenv \
  httpx \
  feedparser \
  pandas \
  openpyxl \
  tqdm \
  gspread \
  google-auth \
  google-auth-oauthlib \
  google-auth-httplib2 \
  google-api-python-client

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 57.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 4.8 MB/s eta 0:00:00


### To run this, we need a groq key
### Step by step guide to get a free access key:
1.   Enter this website https://console.groq.com/home
2.   Create an account (avoid hotmail, outlook or live because it won't work)
3. go to API Keys > create API Key > Save the key somewhere you have access to

In [ ]:
# Enter Groq API key
import os
from getpass import getpass

# Hidden input — nothing is shown while typing
os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")

print("Key loaded successfully ✅")

Enter your Groq API key: ··········
Key loaded successfully ✅


#1. Fetcher

In [ ]:
import os
import csv
import time
import random
from datetime import datetime, timedelta
from typing import List, Dict, Optional, Sequence

import httpx
import feedparser

ARXIV_API = "https://export.arxiv.org/api/query"

DEFAULT_CATEGORIES = [
    # Core ML / AI
    "cs.AI",
    "cs.LG",
    "stat.ML",
    # Medical imaging
    "cs.CV",
    "eess.IV",
    # Biosignals
    "eess.SP",
    # Biomedical / genomics
    "q-bio.GN",
    "q-bio.NC",
    "q-bio.BM",
    "q-bio.TO",
    # Medical physics
    "physics.med-ph",
    "physics.bio-ph",
    # Epidemiology / stats
    "math.ST",
]

# -------------------------
# Helpers: dates + queries
# -------------------------
def _to_arxiv_dt(dt: datetime) -> str:
    """arXiv date format: YYYYMMDDHHMM"""
    return dt.strftime("%Y%m%d%H%M")


def _parse_yyyy_mm_dd(s: str) -> datetime:
    """Parse 'YYYY-MM-DD' into datetime at midnight."""
    return datetime.strptime(s, "%Y-%m-%d")


def _build_query(categories: Sequence[str], date_from: Optional[str], date_to: Optional[str]) -> str:
    cat_part = "(" + " OR ".join([f"cat:{c}" for c in categories]) + ")"
    if date_from and date_to:
        return f"{cat_part} AND submittedDate:[{date_from} TO {date_to}]"
    return cat_part


# Entry normalization
# -------------------------
def _parse_entry(entry) -> Dict:
    entry_id = entry.get("id", "") or ""
    arxiv_id = entry_id.split("/abs/")[-1] if "/abs/" in entry_id else entry_id.split("/")[-1]

    pdf_url = ""
    for link in entry.get("links", []) or []:
        href = getattr(link, "href", "") or ""
        ltype = getattr(link, "type", "") or ""
        if ltype == "application/pdf":
            pdf_url = href
            break

    categories = []
    for t in (entry.get("tags") or []):
        term = None
        if isinstance(t, dict):
            term = t.get("term")
        else:
            term = getattr(t, "term", None)
        if term:
            categories.append(term)

    return {
        "arxiv_id": arxiv_id,
        "id": entry_id,
        "title": (entry.get("title", "") or "").strip().replace("\n", " "),
        "abstract": (entry.get("summary", "") or "").strip().replace("\n", " "),
        "authors": [a.name for a in entry.get("authors", [])] if entry.get("authors") else [],
        "published": entry.get("published", "") or "",
        "updated": entry.get("updated", "") or "",
        "link": entry.get("link", "") or entry_id,
        "pdf_url": pdf_url,
        "categories": categories,
    }

# Robust HTTP fetch w/ retry
# -------------------------
def _get_with_retry(
    client: httpx.Client,
    params: Dict,
    max_attempts: int = 5,
    base_sleep: float = 3.5,
) -> str:
    last_err = None

    for attempt in range(1, max_attempts + 1):
        try:
            r = client.get(ARXIV_API, params=params)
            r.raise_for_status()
            return r.text

        except httpx.HTTPStatusError as e:
            last_err = e
            status = e.response.status_code

            # If arXiv rate-limits us, back off harder
            if status == 429:
                sleep_s = base_sleep * (2 ** (attempt - 1)) + random.uniform(0.5, 1.5)
                print(f"[429] attempt {attempt}/{max_attempts} -> sleeping {sleep_s:.1f}s")
                time.sleep(sleep_s)
            else:
                sleep_s = base_sleep + random.uniform(0.25, 0.75)
                print(f"[HTTP {status}] attempt {attempt}/{max_attempts} -> sleeping {sleep_s:.1f}s")
                time.sleep(sleep_s)

        except (httpx.TimeoutException, httpx.TransportError) as e:
            last_err = e
            sleep_s = base_sleep * (2 ** (attempt - 1)) + random.uniform(0.5, 1.5)
            print(f"[Network retry] attempt {attempt}/{max_attempts} -> sleeping {sleep_s:.1f}s")
            time.sleep(sleep_s)

        except Exception as e:
            last_err = e
            sleep_s = base_sleep * (2 ** (attempt - 1)) + random.uniform(0.5, 1.5)
            print(f"[Retry] attempt {attempt}/{max_attempts} -> sleeping {sleep_s:.1f}s")
            time.sleep(sleep_s)

    raise RuntimeError(f"Failed after {max_attempts} attempts. Last error: {last_err}")


# Main fetcher
# -------------------------
def fetch_papers_by_date_range(
    start_date: str,               # 'YYYY-MM-DD'
    end_date: str,                 # 'YYYY-MM-DD' (inclusive)
    categories: Optional[List[str]] = None,
    max_results_total: int = 500,
    page_size: int = 100,
    chunk_days: int = 1,
    timeout: int = 30,
    polite_delay_seconds: float = 3.5,
) -> List[Dict]:
    """
    Fetch arXiv papers for chosen categories within a date range.
    Returns: list of normalized paper dicts (deduped).
    """
    cats = categories or DEFAULT_CATEGORIES

    start_dt = _parse_yyyy_mm_dd(start_date)
    end_dt = _parse_yyyy_mm_dd(end_date)
    end_dt_inclusive = end_dt.replace(hour=23, minute=59)

    seen = set()
    out: List[Dict] = []

    headers = {
        "User-Agent": "nimbleminds-capstone-arxiv-fetcher/1.0 (contact: your_email@example.com)"
    }

    with httpx.Client(timeout=timeout, headers=headers) as client:
        window_start = start_dt

        while window_start <= end_dt_inclusive and len(out) < max_results_total:
            window_end = min(window_start + timedelta(days=chunk_days) - timedelta(minutes=1), end_dt_inclusive)

            date_from = _to_arxiv_dt(window_start)
            date_to = _to_arxiv_dt(window_end)
            query = _build_query(cats, date_from=date_from, date_to=date_to)

            start = 0
            while len(out) < max_results_total:
                batch = min(page_size, max_results_total - len(out))
                params = {
                    "search_query": query,
                    "start": start,
                    "max_results": batch,
                    "sortBy": "submittedDate",
                    "sortOrder": "descending",
                }

                xml = _get_with_retry(client, params=params)
                feed = feedparser.parse(xml)
                entries = feed.entries or []
                if not entries:
                    break

                for entry in entries:
                    paper = _parse_entry(entry)
                    key = paper.get("arxiv_id") or paper.get("id")
                    if key and key not in seen:
                        seen.add(key)
                        out.append(paper)

                if len(entries) < batch:
                    break

                start += batch
                if polite_delay_seconds:
                    time.sleep(polite_delay_seconds)

            print(f"[Fetcher] {window_start.date()} → {window_end.date()} | pulled so far: {len(out)}")
            if polite_delay_seconds:
                time.sleep(polite_delay_seconds)
            window_start = window_start + timedelta(days=chunk_days)

    print(f"[Fetcher] DONE | Total unique papers: {len(out)}")
    return out

# Saving fetched outputs (CSV only)
# -------------------------
def save_fetched_outputs(
    papers: List[Dict],
    start_date: str,
    end_date: str,
    base_folder: str = "Fetched",
    csv_name: str = "arxiv_fetched.csv",
) -> str:
    """
    Creates:
      Fetched/{start_date}:{end_date}/
        arxiv_fetched.csv

    Returns the output folder path.
    """
    run_folder = os.path.join(base_folder, f"{start_date}:{end_date}")
    os.makedirs(run_folder, exist_ok=True)

    if not papers:
        print(f"[Save] No papers to save in {run_folder}")
        return run_folder

    csv_path = os.path.join(run_folder, csv_name)

    # Union of all keys so we don't miss columns
    fieldnames = sorted({k for p in papers for k in p.keys()})

    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(papers)

    print(f"[Save] Wrote {len(papers)} rows -> {csv_path}")
    return run_folder

# Example usage
# -------------------------
if __name__ == "__main__":
    start_date = "2023-01-01"
    end_date = "2023-01-31"

    papers = fetch_papers_by_date_range(
        start_date=start_date,
        end_date=end_date,
        max_results_total=100,
        chunk_days=1,
    )

    # Print a quick sanity check
    #for p in papers:
    #    primary_category = p.get("categories", ["unknown"])[0]
    #    section = primary_category.split(".")[0]
    #    print("\nSECTION:", section)
    #    print("CATEGORY:", primary_category)
    #    print("TITLE:", p.get("title", ""))
    #    print("ABSTRACT:", (p.get("abstract", "") or "")[:200])
    #    print("-" * 80)

[Fetcher] 2023-01-01 → 2023-01-01 | pulled so far: 65
[Fetcher] 2023-01-02 → 2023-01-02 | pulled so far: 100
[Fetcher] DONE | Total unique papers: 100


#2. Classifier

In [ ]:
from groq import Groq
# To clean the key and not have an error
raw_api_key = os.getenv("GROQ_API_KEY")
if raw_api_key is None:
    raise ValueError("GROQ_API_KEY is not set.")

clean_api_key = (
    str(raw_api_key)
    .replace("\u2028", "")
    .replace("\u2029", "")
    .replace("\n", "")
    .replace("\r", "")
    .strip()
)

client = Groq(api_key=clean_api_key)


print("Cleaned repr")
print("Length:", len(os.getenv("GROQ_API_KEY")))
print("Non-ASCII chars:", [(i, ch, hex(ord(ch))) for i, ch in enumerate(os.getenv("GROQ_API_KEY") or "") if ord(ch) > 127])

Cleaned repr
Length: 57
Non-ASCII chars: [(56, '\u2028', '0x2028')]


In [ ]:
import os
import re
import json
import time
import random
from typing import Dict, List, Tuple, Any, Optional
import unicodedata

from groq import Groq
from tqdm.auto import tqdm

# Groq client
# -------------------------
MODEL_NAME = os.getenv("GROQ_MODEL", "llama-3.3-70b-versatile")
client = Groq(api_key=clean_api_key)

# JSON parsing + helpers
# -------------------------
def _safe_json_load(s: str) -> Dict[str, Any]:
    """Try to parse JSON even if model adds extra text/code fences."""
    s = (s or "").strip()

    # Strip code fences
    if s.startswith("```"):
        s = re.sub(r"^```[a-zA-Z]*\n?", "", s)
        s = s.replace("```", "").strip()

    # Attempt direct parse
    try:
        return json.loads(s)
    except Exception:
        pass

    # Try to extract first {...} block
    start = s.find("{")
    end = s.rfind("}")
    if start != -1 and end != -1 and end > start:
        return json.loads(s[start : end + 1])

    raise ValueError(f"Could not parse JSON from model output:\n{s}")


def _truncate(text: str, max_chars: int) -> str:
    text = (text or "").strip()
    if len(text) <= max_chars:
        return text
    return text[:max_chars].rsplit(" ", 1)[0] + "..."


LLM_CALLS = 0  # shows up in progress bar

def _show_non_ascii(label: str, text: str):
    bad = [(i, ch, hex(ord(ch))) for i, ch in enumerate(text) if ord(ch) > 127]
    if bad:
        print(f"[NON-ASCII FOUND] {label}")
        for item in bad[:20]:
            print(item)

def _groq_json(prompt: str, max_attempts: int = 3, base_sleep: float = 1.0) -> Dict[str, Any]:
    global LLM_CALLS
    last_err: Optional[Exception] = None

    system_msg = _clean_text("Return STRICT JSON only. No markdown. No extra text.")
    user_msg = _clean_text(prompt)
    _show_non_ascii("SYSTEM MSG", system_msg)
    _show_non_ascii("USER MSG", user_msg)

    for attempt in range(1, max_attempts + 1):
        try:
            LLM_CALLS += 1
            resp = client.chat.completions.create(
                model=MODEL_NAME,
                temperature=0,
                messages=[
                    {"role": "system", "content": system_msg},
                    {"role": "user", "content": user_msg},
                ],
            )
            raw = _clean_text(resp.choices[0].message.content or "")
            return _safe_json_load(raw)
        except Exception as e:
            last_err = e
            time.sleep(base_sleep * (2 ** (attempt - 1)) + random.random() * 0.25)

    raise RuntimeError(f"Groq call failed after {max_attempts} attempts. Last error: {last_err}")

# 1) Keyword prefilter: Healthcare hints (*Recommend adding a few radiology/oncology/Pathology/general medicine terms to HEALTHCARE_HINTS*)
# -----------------------------------------------------
HEALTHCARE_HINTS = [
    "patient", "patients", "clinical", "clinician", "hospital", "diagnosis", "treatment",
    "disease", "screening", "prognosis", "therapy", "surgery", "mortality", "morbidity",
    "epidemiology", "public health", "cohort", "randomized", "trial", "ICU", "intensive care",
    "mri", "ct", "ultrasound", "x-ray", "radiology", "pathology",
    "ecg", "eeg", "ppg", "ehr", "electronic health record", "medical imaging",
]

HEALTHCARE_HINT_PATTERN = re.compile(
    r"\b(" + "|".join(re.escape(k) for k in HEALTHCARE_HINTS) + r")\b",
    re.IGNORECASE
)


def keyword_healthcare_hits(title: str, abstract: str) -> List[str]:
    text = f"{title}\n{abstract}"
    return sorted(set(m.group(0) for m in HEALTHCARE_HINT_PATTERN.finditer(text)), key=str.lower)


# 2) Specialty keyword libraries (SCALABLE)
#   Outputs for now: Radiology, Oncology, Other
# ---------------------------------------------------------------------------------
RADIOLOGY_KEYWORDS = [
    "Radiology", "Radiologic imaging", "Radiological imaging", "Radiologist", "Diagnostic imaging",
    #"Medical imaging",
    "Radiography", "X-ray", "Computed tomography","Clinical Imaging",
    #"CT",
    "CT scan","CT scans", "Magnetic resonance imaging", "MRI", "MRI scan","MRI scans",
    "Ultrasound", "Ultrasonography", "Sonography", "Fluoroscopy",
    "Mammography", "CTA", "CT Angiography", "MR Angiography", "Interventional radiology",
    "Neuroradiology", "Musculoskeletal imaging", "Breast imaging",
    "Chest imaging", "Abdominal imaging", "Pelvic imaging", "Radiologists",
    "Thoracic imaging", "Cardiac imaging", "Vascular imaging",
    "Nuclear imaging", "Positron emission tomography", #"PET", "PET-CT",
    ## New words underneath
    "CT scan", "Computed tomography","PET scan", "PET imaging",
    "image segmentation", "image reconstruction", "radiomics","dicom", "imaging pipeline",
    "medical image analysis", "image analysis", "image segmentation", "deep learning for imaging",
    "organ segmentation", "tumor segmentation", "medical segmentation", "segmentation model",
    "segmentation models", "CT-based segmentation", "volumetric segmentation", "3D segmentation",
    "computed tomography imaging", "medical image segmentation"
    ## End of New words
    "Cross-sectional imaging", "Functional imaging", "Molecular imaging",
    "Imaging findings", "Radiologic findings", "Radiological findings",
    "Imaging modality", "Imaging modalities", "Imaging techniques",
    "Image interpretation", "Radiologic diagnosis", "Radiological evaluation",
    "Radiologic assessment", "Radiologist", "Screening imaging",
    "Tumor imaging", "Cancer imaging", "Brain imaging", "Spinal imaging",
    "Head and neck imaging", "Whole-body imaging", "Contrast-enhanced imaging",
    "Contrast media", "Image-guided diagnosis", "Image acquisition",
    "Image reconstruction", "Imaging biomarkers", "Diagnostic radiology",
    "Emergency radiology", "Pediatric radiology", "Body imaging", "Brain MRI", "Neuroimaging",
    "Diffusion-weighted imaging", "DWI"
]

ONCOLOGY_KEYWORDS = [
    "Oncology", "Cancer",# "Tumor", "Tumour", "Neoplasm", "Neoplasia",
    "Malignant tumor", "Solid tumor", "Hematologic malignancy", "Malignancy",
    "Carcinoma", "Sarcoma", "Lymphoma", "Leukemia", "Myeloma",
    "Metastasis", "Metastatic cancer", #"Primary tumor", "Secondary tumor", "Benign tumor",
    "Tumor progression", "Tumor growth", "Tumor burden", "Tumor heterogeneity",
    "Tumor microenvironment", "Cancer biology", "Cancer research",
    "Clinical oncology", "Medical oncology", "Surgical oncology", "Radiation oncology",
    "Pediatric oncology", "Translational oncology", "Precision oncology",
    "Molecular oncology", "Experimental oncology", "Breast cancer", "Lung cancer", "Prostate cancer", "Colorectal cancer",
    "Pancreatic cancer", "Liver cancer", "Hepatocellular carcinoma",
    "Gastric cancer", "Esophageal cancer", "Ovarian cancer", "Cervical cancer",
    "Endometrial cancer", "Bladder cancer", "Kidney cancer", "Renal cell carcinoma",
    "Brain tumor", "Glioma", "Glioblastoma", "Melanoma", "Skin cancer",
    "Thyroid cancer", "Head and neck cancer", "Chemotherapy", "Radiotherapy", "Radiation therapy",
    #"Targeted therapy","Immunotherapy", "Hormone therapy", "Endocrine therapy",
    "Cancer treatment", "Cancer therapy", #"Combination therapy",
    #"Adjuvant therapy", "Neoadjuvant therapy", "Palliative care",
    "CAR-T therapy", "Checkpoint inhibitors", #"PD-1", "PD-L1", "CTLA-4",
    #"Drug resistance",
    "Cancer drug development", "Cancer diagnosis", "Cancer screening", #"Early detection","Tumor detection",
    "Cancer prognosis",# "Survival analysis", "Disease progression",
    #"Recurrence", "Remission", "Clinical trials", "Phase I trial",
    #"Phase II trial", "Phase III trial", "Randomized controlled trial",
    "Oncology outcomes", "Oncogene", "Tumor suppressor gene",
    #"p53", "KRAS", "EGFR", "BRAF",
    "Cancer genomics", #"Genomic profiling", "Transcriptomics", "Proteomics",
    #"Biomarkers",
    "Tumor biomarkers", "Liquid biopsy",
    "Circulating tumor DNA", "ctDNA", "Cancer stem cells",
    #"Cell proliferation", "Apoptosis", "Angiogenesis",
    "Immune evasion", "Cancer incidence", "Cancer prevalence", "Cancer mortality", #"Risk factors",
    "Carcinogenesis", #"Environmental exposure", "Lifestyle factors",
    "Smoking-related cancer", "Cancer epidemiology", "Cancer disparities",
    "Global cancer burden", #"Biopsy", "Histopathology", "Cytology",
    "Tumor grading", "Tumor staging",
    "TNM staging", "Tumor markers", #"Disease monitoring","Minimal residual disease",
    "AI in oncology", "Machine learning cancer", "Deep learning oncology", "Immuno-oncology"
    #"Digital pathology",
    #"Multi-omics", "Epigenetics", "DNA methylation",
    ## New words added
    "neoadjuvant chemotherapy", "neoadjuvant chemoradiotherapy", "adjuvant chemotherapy", "adjuvant radiotherapy",
    "palliative chemotherapy",
]

PATHOLOGY_KEYWORDS= [
    "Biopsy", "Biopsies", "Core needle biopsy", "Breast biopsy", "Tissue biopsy",
    "Specimen analysis", "Tissue", "Tissue sample", "Whole slide imaging", "Whole slide images",
    "Slide imaging", "Glass slide", "Microscopy", "Digital microscopy", "Pathologist", "Pathologists",
    "Pathology report", "Histopathological analysis", "Histopathological evaluation", #"Diagnosis", "Diagnostic",
    #"Diagnostic accuracy", "Diagnostic assessment", "Diagnostic interpretation", "Tumor pathology",
    "Cancer pathology", "Lymphoma pathology", "Breast pathology", "Immunohistochemistry", "IHC", "Staining",
    "Tissue staining", "Cell morphology", "Morphological analysis", "Cellular features", "Tissue architecture",
    #"Image analysis",
    ## New words added
    "histology", "whole slide image", "WSI", "tissue section", "slide analysis"
    ## End of new words
    ##"Image classification", "Feature extraction", "Pattern recognition",
    "Artificial intelligence in pathology", "Machine learning pathology", "Deep learning pathology",
    #"Prediction", "Prognosis", "Risk stratification",
    "Workflow pathology", "Laboratory pathology", "Pathology workflow"
]
#We can add more lists as we add more specialty keywords
# COLON_RECTAL_SURGERY_KEYWORDS = [ .... ]

SPECIALTY_KEYWORDS: Dict[str, List[str]] = {
    "Radiology": RADIOLOGY_KEYWORDS,
    "Oncology": ONCOLOGY_KEYWORDS,
    "Pathology": PATHOLOGY_KEYWORDS
    # Scalable as we can add more specialties over here
}

OUTPUT_SPECIALTIES = ["Radiology","Oncology","Pathology", "Other"]
#Add other specialties here

def _compile_specialty_patterns(spec_kw: Dict[str, List[str]]) -> Dict[str, re.Pattern]:
    patterns: Dict[str, re.Pattern] = {}
    for specialty, keywords in spec_kw.items():
        seen = set()
        deduped: List[str] = []
        for k in keywords:
            kk = (k or "").strip()
            if kk and kk.lower() not in seen:
                deduped.append(kk)
                seen.add(kk.lower())
        if deduped:
            patterns[specialty] = re.compile("|".join(re.escape(k) for k in deduped), re.IGNORECASE)
    return patterns

# DEBUG BLOCK
for specialty, keywords in SPECIALTY_KEYWORDS.items():
    print(f"\n--- {specialty} ---")
    print("Type of keywords object:", type(keywords))
    for i, k in enumerate(keywords):
        if not isinstance(k, str):
            print(f"Bad item at index {i}: {k} | type={type(k)}")


SPECIALTY_PATTERNS = _compile_specialty_patterns(SPECIALTY_KEYWORDS)


def score_specialty(title: str, abstract: str) -> Dict[str, Any]:
    """
    Keyword-only specialty classifier:
    - returns best specialty among the 3 if score>0
    - else "Other"
    Also returns matches for audit.
    """
    text = f"{title}\n{abstract}"
    best_spec = "Other"
    best_score = 0
    best_matches: List[str] = []

    for spec, pattern in SPECIALTY_PATTERNS.items():
        matches = sorted(set(m.group(0) for m in pattern.finditer(text)), key=str.lower)
        score = len(matches)
        if score > best_score:
            best_score = score
            best_spec = spec
            best_matches = matches

    return {
        "specialty": best_spec if best_score > 0 else "Other",
        "keyword_score": best_score,
        "keyword_matches": best_matches[:25],
    }

def _clean_text(text: Any) -> str:
    if text is None:
        return ""

    text = str(text)

    replacements = {
        "\u2028": " ",
        "\u2029": " ",
        "\xa0": " ",
        "–": "-",
        "—": "-",
        "“": '"',
        "”": '"',
        "’": "'",
        "‘": "'",
    }
    for old, new in replacements.items():
        text = text.replace(old, new)

    text = unicodedata.normalize("NFKD", text)
    text = text.encode("ascii", "ignore").decode("ascii")
    text = re.sub(r"\s+", " ", text).strip()
    return text

# 3) LLM Healthcare Gate
# -------------------------
def llm_is_healthcare(title: str, abstract: str, kw_hits: List[str]) -> Dict[str, Any]:
    title = _clean_text(title)
    abstract = _clean_text(abstract)
    kw_hits = [_clean_text(k) for k in kw_hits]
    abstract_t = _truncate(abstract, 1400)

    prompt_lines = [
        "Label whether this arXiv paper is healthcare/medicine related.",
        "",
        "Definition of HEALTHCARE:",
        "- human health, clinical medicine, disease, diagnosis, treatment, patient care, healthcare systems,",
        "  public health/epidemiology, medical imaging used for diagnosis, medical devices used in care.",
        "",
        "NOT healthcare unless explicitly tied to human health/clinical context:",
        "- generic ML/optimization, robotics/control, pure neuroscience/cognition without clinical tie,",
        "- agriculture/plant biology/ecology, general biology without disease/clinical relevance.",
        "",
        f"Keyword hints found (may help, but do not over-trust): {json.dumps(kw_hits)}",
        "",
        "Return STRICT JSON:",
        "{",
        '  "is_healthcare": true,',
        '  "confidence": 0.0,',
        '  "reason": "short evidence-based reason referencing phrases from title/abstract"',
        "}",
        "",
        f"TITLE: {title}",
        f"ABSTRACT: {abstract_t}",
    ]
    _show_non_ascii("TITLE", title)
    _show_non_ascii("ABSTRACT", abstract_t)
    prompt = _clean_text("\n".join(prompt_lines))
    data = _groq_json(prompt)

    return {
        "is_healthcare": bool(data.get("is_healthcare", False)),
        "confidence": float(data.get("confidence", 0.0)),
        "reason": _clean_text(str(data.get("reason", "")).strip()),
    }

# 4) Full classifier for one paper (scalable)
# -----------------------------------------------------
def classify_paper(paper: Dict[str, Any], llm_threshold: float = 0.55) -> Dict[str, Any]:
    """
    Expects paper fields from your fetcher:
      - title
      - abstract   (IMPORTANT: your fetcher uses "abstract")
      - categories (list)
      - plus any metadata (arxiv_id, link, pdf_url, ...)
    """
    title = _clean_text(paper.get("title", "") or "")
    abstract = _clean_text(paper.get("abstract", "") or "")

    kw_hits = keyword_healthcare_hits(title, abstract)

    # Optimization: if zero hints, skip LLM to save calls (conservative)
    if not kw_hits:
        hc = {
            "is_healthcare": False,
            "confidence": 0.35,
            "reason": "No healthcare keyword hints; skipped LLM gate.",
        }
    else:
        hc = llm_is_healthcare(title, abstract, kw_hits)
        # Optional conservative threshold
        if hc["is_healthcare"] and hc["confidence"] < llm_threshold:
            hc["is_healthcare"] = False
            hc["reason"] = f"Low-confidence healthcare ({hc['confidence']}); treated as non-healthcare. " + hc["reason"]

    # Specialty only if healthcare
    if hc["is_healthcare"]:
        spec = score_specialty(title, abstract)
    else:
        spec = {"specialty": "Other", "keyword_score": 0, "keyword_matches": []}

    out = dict(paper)
    out.update({
        "healthcare": hc["is_healthcare"],
        "hc_confidence": hc["confidence"],
        "hc_reason": hc["reason"],
        "hc_keyword_hits": "|".join(kw_hits[:25]),
        "specialty": spec["specialty"],
        "spec_keyword_score": spec["keyword_score"],
        "spec_keyword_matches": "|".join(spec["keyword_matches"]),
    })
    return out


# 5) Batch runner + progress bar + summary
# -----------------------------------------
def classify_batch(papers: List[Dict[str, Any]]) -> Tuple[List[Dict[str, Any]], Dict[str, Any]]:
    rows: List[Dict[str, Any]] = []
    total = len(papers)
    hc_count = 0
    spec_counts: Dict[str, int] = {s: 0 for s in OUTPUT_SPECIALTIES}

    bar = tqdm(papers, desc="Classifying papers", unit="paper")
    for i, p in enumerate(bar, start=1):
        try:
            row = classify_paper(p)
            rows.append(row)

            if row["healthcare"]:
                hc_count += 1
                spec = row["specialty"]
                if spec not in spec_counts:
                    spec = "Other"
                spec_counts[spec] += 1

            bar.set_postfix(llm_calls=LLM_CALLS, healthcare=hc_count)

        except Exception as e:
            print(f"\n[ERROR] Failed on paper #{i}")
            print("RAW TITLE:", repr(p.get("title", "")))
            print("RAW ABSTRACT:", repr((p.get("abstract", "") or "")[:500]))
            raise

    summary = {
        "total": total,
        "healthcare": hc_count,
        "healthcare_rate": (hc_count / total) if total else 0.0,
        "specialties": spec_counts,
    }
    return rows, summary


--- Radiology ---
Type of keywords object: <class 'list'>

--- Oncology ---
Type of keywords object: <class 'list'>

--- Pathology ---
Type of keywords object: <class 'list'>


In [ ]:
# To share the folders to our google drive, we need to give permission to access
# run the code below to give that access
# Make the path for the shared AI file on your drive to look similar to this below
# content/drive/MyDrive/AI/...

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

BASE_DIR = "/content/drive/MyDrive/AI/Scraping Agent/Output"

FETCHED_DIR = os.path.join(BASE_DIR, "Fetched")
CLASSIFIED_DIR = os.path.join(BASE_DIR, "Classified")

os.makedirs(FETCHED_DIR, exist_ok=True)
os.makedirs(CLASSIFIED_DIR, exist_ok=True)

print("Created folders:")
print(FETCHED_DIR)
print(CLASSIFIED_DIR)

Created folders:
/content/drive/MyDrive/AI/Scraping Agent/Output/Fetched
/content/drive/MyDrive/AI/Scraping Agent/Output/Classified


# 3. Storage

In [ ]:
import os
import csv
from collections import defaultdict
from typing import List, Dict, Any

def save_rows_to_csv(rows: List[Dict[str, Any]], path: str):
    os.makedirs(os.path.dirname(path), exist_ok=True)

    if not rows:
        print(f"[Save] No rows to save -> {path}")
        return False

    fieldnames = sorted({k for r in rows for k in r.keys()})

    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

    exists = os.path.exists(path)
    size = os.path.getsize(path) if exists else 0

    if not exists or size == 0:
        raise RuntimeError(f"File write failed or empty file: {path}")

    print(f"[Save] Wrote {len(rows)} rows -> {path} (exists={exists}, size={size})")
    return True

def _slugify(name: str) -> str:
    import re
    name = (name or "").strip().lower()
    name = re.sub(r"[^a-z0-9]+", "_", name)
    return name.strip("_") or "other"


from collections import defaultdict
import os

def save_specialty_outputs(
    rows: List[Dict[str, Any]],
    base_dir: str,
    start_date: str = "",
    end_date: str = "",
):
    os.makedirs(base_dir, exist_ok=True)

    healthcare_rows = [r for r in rows if r.get("healthcare")]
    print(f"[Debug] healthcare_rows = {len(healthcare_rows)}")

    # Save master classified file
    master_path = os.path.join(base_dir, f"{start_date}_{end_date}_specialties_master.csv")
    save_rows_to_csv(healthcare_rows, master_path)

    # Save one file per specialty
    buckets = defaultdict(list)
    for r in healthcare_rows:
        specialty = r.get("specialty", "Other")
        buckets[specialty].append(r)

    for specialty, spec_rows in buckets.items():
        print(f"[Debug] Saving specialty = {specialty}, rows = {len(spec_rows)}")

        folder = os.path.join(base_dir, _slugify(specialty))
        os.makedirs(folder, exist_ok=True)

        out_path = os.path.join(folder, f"{start_date}_{end_date}_{_slugify(specialty)}.csv")
        save_rows_to_csv(spec_rows, out_path)

def save_run_summary(summary: Dict[str, Any], output_path: str):
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    with open(output_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["metric", "value"])
        for k, v in summary.items():
            writer.writerow([k, v])

    print(f"[Save] Wrote summary -> {output_path}")

# helper so that after classification, loop through the healthcare rows and extract full text
def save_full_texts_by_specialty(rows: List[Dict[str, Any]], base_dir: str):
    for row in rows:
        if not row.get("healthcare"):
            continue

        if already_saved_in_specialty(row, base_dir):
            print(f"[Skip] Already saved full text for {row.get('arxiv_id')}")
            continue

        pdf_url = row.get("pdf_url", "")
        if not pdf_url:
            print(f"[Skip] No PDF URL for {row.get('arxiv_id')}")
            continue

        try:
            full_text = extract_text_from_pdf_url(pdf_url)
            abstract_pdf = extract_abstract_from_pdf_url(pdf_url)

            row["abstract_pdf"] = abstract_pdf

            append_full_text_to_specialty_file(row, full_text, base_dir)
            mark_saved_in_specialty(row, base_dir)
        except Exception as e:
            print(f"[Error] Failed full-text extraction for {row.get('arxiv_id')}: {e}")

In [ ]:
## Helper to extract from PDF
import fitz  # PyMuPDF
import httpx
import os

def extract_text_from_pdf_url(pdf_url: str, temp_pdf_path: str = "/tmp/temp_paper.pdf") -> str:
    if not pdf_url:
        return ""

    r = httpx.get(pdf_url, timeout=60)
    r.raise_for_status()

    with open(temp_pdf_path, "wb") as f:
        f.write(r.content)

    text_parts = []
    doc = fitz.open(temp_pdf_path)
    for page in doc:
        text_parts.append(page.get_text())
    doc.close()

    return "\n".join(text_parts).strip()

## Helper to extract from PDF
import fitz  # PyMuPDF
import httpx
import os

def extract_text_from_pdf_url(pdf_url: str, temp_pdf_path: str = "/tmp/temp_paper.pdf") -> str:
    if not pdf_url:
        return ""

    r = httpx.get(pdf_url, timeout=60)
    r.raise_for_status()

    with open(temp_pdf_path, "wb") as f:
        f.write(r.content)

    text_parts = []
    doc = fitz.open(temp_pdf_path)
    for page in doc:
        text_parts.append(page.get_text())
    doc.close()

    return "\n".join(text_parts).strip()

import re
import fitz
import httpx

def _clean_pdf_text(text: str) -> str:
    if not text:
        return ""

    # fix hyphenation across line breaks
    text = re.sub(r"(\w)-\s+(\w)", r"\1\2", text)

    # normalize whitespace
    text = text.replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{2,}", "\n\n", text)
    return text.strip()


def _page_text_in_reading_order(page) -> str:
    """
    Extract text blocks and sort them in reading order:
    top-to-bottom, then left-to-right.
    This is more reliable than page.get_text() for multi-column PDFs.
    """
    blocks = page.get_text("blocks")
    # block tuple: (x0, y0, x1, y1, text, block_no, block_type, ...)
    text_blocks = [b for b in blocks if len(b) >= 5 and isinstance(b[4], str) and b[4].strip()]
    text_blocks.sort(key=lambda b: (round(b[1], 1), round(b[0], 1)))
    return "\n".join(b[4].strip() for b in text_blocks)


def _extract_abstract_from_text(full_text: str) -> str:
    if not full_text:
        return ""

    text = _clean_pdf_text(full_text)

    # Start at Abstract / ABSTRACT / Abstract—
    start_match = re.search(r"\bAbstract\b\s*[\.\-—:]?\s*", text, flags=re.IGNORECASE)
    if not start_match:
        return ""

    text = text[start_match.end():]

    # Stop at first major section heading, but DO NOT stop at
    # Objective / Approach / Main Results / Significance
    end_patterns = [
        r"\n\s*Keywords\b",
        r"\n\s*Index Terms\b",
        r"\n\s*I\.\s*INTRODUCTION\b",
        r"\n\s*1\.\s*Introduction\b",
        r"\n\s*1\s+Introduction\b",
        r"\n\s*Introduction\b",
        r"\n\s*II\.\s*",
        r"\n\s*2\.\s*",
    ]

    end_positions = []
    for pattern in end_patterns:
        m = re.search(pattern, text, flags=re.IGNORECASE)
        if m:
            end_positions.append(m.start())

    if end_positions:
        text = text[:min(end_positions)]

    text = _clean_pdf_text(text)

    # Final flattening for one-paragraph abstract storage
    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text).strip()

    return text


def extract_abstract_from_pdf_url(
    pdf_url: str,
    temp_pdf_path: str = "/tmp/temp_paper.pdf",
    max_pages: int = 2,
) -> str:
    if not pdf_url:
        return ""

    r = httpx.get(pdf_url, timeout=60)
    r.raise_for_status()

    with open(temp_pdf_path, "wb") as f:
        f.write(r.content)

    doc = fitz.open(temp_pdf_path)
    try:
        pages_text = []
        for i, page in enumerate(doc):
            if i >= max_pages:
                break
            pages_text.append(_page_text_in_reading_order(page))

        first_pages_text = "\n\n".join(pages_text)
        return _extract_abstract_from_text(first_pages_text)
    finally:
        doc.close()

# Helper to append article text to specialty
def append_full_text_to_specialty_file(
    paper: Dict[str, Any],
    full_text: str,
    base_dir: str,
):
    specialty = paper.get("specialty", "Other")
    folder_name = _slugify(specialty)
    folder = os.path.join(base_dir, folder_name)
    os.makedirs(folder, exist_ok=True)

    txt_path = os.path.join(folder, f"{folder_name}_full_articles.txt")

    with open(txt_path, "a", encoding="utf-8") as f:
        f.write("=" * 60 + "\n")
        f.write(f"ARXIV ID: {paper.get('arxiv_id', '')}\n")
        f.write(f"TITLE: {paper.get('title', '')}\n")
        f.write(f"SPECIALTY: {specialty}\n")
        f.write(f"PDF URL: {paper.get('pdf_url', '')}\n")
        f.write(f"AUTHORS: {', '.join(paper.get('authors', []))}\n")
        f.write(f"PUBLISHED: {paper.get('published', '')}\n")
        f.write(f"ABSTRACT PDF: {paper.get('abstract_pdf', '')}\n")
        f.write("=" * 60 + "\n\n")
        f.write("FULL TEXT:\n")
        f.write(full_text or "[NO TEXT EXTRACTED]")
        f.write("\n\nEND OF ARTICLE\n")
        f.write("#" * 60 + "\n\n")

    print(f"[Save] Appended article text -> {txt_path}")

# Helper to avoid duplicates
def already_saved_in_specialty(paper: Dict[str, Any], base_dir: str) -> bool:
    specialty = paper.get("specialty", "Other")
    folder_name = _slugify(specialty)
    folder = os.path.join(base_dir, folder_name)
    os.makedirs(folder, exist_ok=True)

    id_log_path = os.path.join(folder, f"{folder_name}_saved_ids.txt")
    arxiv_id = paper.get("arxiv_id", "")

    if not arxiv_id:
        return False

    if os.path.exists(id_log_path):
        with open(id_log_path, "r", encoding="utf-8") as f:
            existing = set(line.strip() for line in f if line.strip())
        return arxiv_id in existing

    return False

def mark_saved_in_specialty(paper: Dict[str, Any], base_dir: str):
    specialty = paper.get("specialty", "Other")
    folder_name = _slugify(specialty)
    folder = os.path.join(base_dir, folder_name)
    os.makedirs(folder, exist_ok=True)

    id_log_path = os.path.join(folder, f"{folder_name}_saved_ids.txt")
    arxiv_id = paper.get("arxiv_id", "")

    if arxiv_id:
        with open(id_log_path, "a", encoding="utf-8") as f:
            f.write(arxiv_id + "\n")

# Helper to append article text to specialty
def append_full_text_to_specialty_file(
    paper: Dict[str, Any],
    full_text: str,
    base_dir: str,
):
    specialty = paper.get("specialty", "Other")
    folder_name = _slugify(specialty)
    folder = os.path.join(base_dir, folder_name)
    os.makedirs(folder, exist_ok=True)

    txt_path = os.path.join(folder, f"{folder_name}_full_articles.txt")

    with open(txt_path, "a", encoding="utf-8") as f:
        f.write("=" * 60 + "\n")
        f.write(f"ARXIV ID: {paper.get('arxiv_id', '')}\n")
        f.write(f"TITLE: {paper.get('title', '')}\n")
        f.write(f"SPECIALTY: {specialty}\n")
        f.write(f"PDF URL: {paper.get('pdf_url', '')}\n")
        f.write(f"AUTHORS: {', '.join(paper.get('authors', []))}\n")
        f.write(f"PUBLISHED: {paper.get('published', '')}\n")
        f.write(f"ABSTRACT PDF: {paper.get('abstract_pdf', '')}\n")
        f.write("=" * 60 + "\n\n")
        f.write("FULL TEXT:\n")
        f.write(full_text or "[NO TEXT EXTRACTED]")
        f.write("\n\nEND OF ARTICLE\n")
        f.write("#" * 60 + "\n\n")

    print(f"[Save] Appended article text -> {txt_path}")

# Helper to avoid duplicates
def already_saved_in_specialty(paper: Dict[str, Any], base_dir: str) -> bool:
    specialty = paper.get("specialty", "Other")
    folder_name = _slugify(specialty)
    folder = os.path.join(base_dir, folder_name)
    os.makedirs(folder, exist_ok=True)

    id_log_path = os.path.join(folder, f"{folder_name}_saved_ids.txt")
    arxiv_id = paper.get("arxiv_id", "")

    if not arxiv_id:
        return False

    if os.path.exists(id_log_path):
        with open(id_log_path, "r", encoding="utf-8") as f:
            existing = set(line.strip() for line in f if line.strip())
        return arxiv_id in existing

    return False


def mark_saved_in_specialty(paper: Dict[str, Any], base_dir: str):
    specialty = paper.get("specialty", "Other")
    folder_name = _slugify(specialty)
    folder = os.path.join(base_dir, folder_name)
    os.makedirs(folder, exist_ok=True)

    id_log_path = os.path.join(folder, f"{folder_name}_saved_ids.txt")
    arxiv_id = paper.get("arxiv_id", "")

    if arxiv_id:
        with open(id_log_path, "a", encoding="utf-8") as f:
            f.write(arxiv_id + "\n")

In [ ]:
import os
import re
import httpx
import fitz
from urllib.parse import urlparse
from typing import List, Dict, Any, Optional

# =========================================================
# METADATA + DATASET + IMAGE EXTRACTION HELPERS
# Paste this AFTER your PDF/helper functions
# and BEFORE run_pipeline()
# =========================================================

DATASET_DOMAINS = [
    "physionet.org",
    "kaggle.com",
    "zenodo.org",
    "figshare.com",
    "datadryad.org",
    "dryad.org",
    "osf.io",
    "huggingface.co",
    "github.com",
    "gitlab.com",
    "openml.org",
    "archive.ics.uci.edu",
    "ncbi.nlm.nih.gov",
    "bioproject.ncbi.nlm.nih.gov",
    "nih.gov",
    "cdc.gov",
    "who.int",
    "data.gov",
]

DATASET_HINT_PATTERNS = [
    r"\bdataset\b",
    r"\bdata set\b",
    r"\bpublic dataset\b",
    r"\bpublicly available dataset\b",
    r"\brepository\b",
    r"\bregistry\b",
    r"\bbiobank\b",
    r"\bcohort\b",
    r"\bdata source\b",
    r"\bavailable at\b",
    r"\bdownloaded from\b",
    r"\bobtained from\b",
    r"\baccessed from\b",
]

URL_PATTERN = re.compile(r'https?://[^\s<>"\)\]]+', re.IGNORECASE)


def clean_url(url: str) -> str:
    return (url or "").strip().rstrip(".,;:)")


def is_dataset_like_url(url: str) -> bool:
    try:
        parsed = urlparse(url)
        netloc = parsed.netloc.lower()
        path = parsed.path.lower()
        full = f"{netloc}{path}"

        if any(domain in netloc for domain in DATASET_DOMAINS):
            return True

        fallback_terms = [
            "dataset", "data", "repository", "record",
            "accession", "download", "cohort", "registry",
        ]
        return any(term in full for term in fallback_terms)
    except Exception:
        return False


def extract_dataset_links_from_text(text: str) -> List[str]:
    urls = [clean_url(m.group(0)) for m in URL_PATTERN.finditer(text or "")]
    urls = list(dict.fromkeys(urls))
    dataset_urls = [u for u in urls if is_dataset_like_url(u)]
    return dataset_urls


def extract_dataset_mentions(text: str, window: int = 120) -> List[str]:
    text = re.sub(r"\s+", " ", text or "").strip()
    mentions = []

    for pattern in DATASET_HINT_PATTERNS:
        for m in re.finditer(pattern, text, flags=re.IGNORECASE):
            start = max(0, m.start() - window)
            end = min(len(text), m.end() + window)
            snippet = text[start:end].strip()
            mentions.append(snippet)

    unique = []
    seen = set()
    for snippet in mentions:
        key = snippet.lower()
        if key not in seen:
            seen.add(key)
            unique.append(snippet)

    return unique


def extract_dataset_info_from_pdf_url(pdf_url: str) -> Dict[str, Any]:
    text = extract_text_from_pdf_url(pdf_url)

    dataset_links = extract_dataset_links_from_text(text)
    dataset_mentions = extract_dataset_mentions(text)

    return {
        "dataset_links": dataset_links,
        "dataset_mentions": dataset_mentions,
    }


def extract_images_from_pdf_url(
    pdf_url: str,
    output_dir: str,
    min_width: int = 100,
    min_height: int = 100,
    min_bytes: int = 5000,
) -> List[str]:
    pdf_bytes = httpx.get(pdf_url, timeout=60).content
    temp_pdf = "/tmp/temp_image_test.pdf"

    with open(temp_pdf, "wb") as f:
        f.write(pdf_bytes)

    os.makedirs(output_dir, exist_ok=True)

    doc = fitz.open(temp_pdf)
    saved_files = []

    try:
        for page_num in range(len(doc)):
            page = doc[page_num]
            image_list = page.get_images(full=True)

            for img_index, img in enumerate(image_list):
                xref = img[0]
                try:
                    base_image = doc.extract_image(xref)
                    image_bytes = base_image["image"]
                    ext = base_image.get("ext", "png")
                    width = base_image.get("width", 0)
                    height = base_image.get("height", 0)

                    if width < min_width or height < min_height:
                        continue
                    if len(image_bytes) < min_bytes:
                        continue

                    filename = f"page_{page_num+1}_img_{img_index+1}.{ext}"
                    filepath = os.path.join(output_dir, filename)

                    if not os.path.exists(filepath):
                        with open(filepath, "wb") as img_file:
                            img_file.write(image_bytes)

                    saved_files.append(filepath)

                except Exception as e:
                    print(
                        f"[Image Extract] Could not extract image on page "
                        f"{page_num+1}, image {img_index+1}: {e}"
                    )
    finally:
        doc.close()

    return saved_files


def safe_folder_name(name: str, max_len: int = 120) -> str:
    name = (name or "").strip()
    name = re.sub(r'[\\/*?:"<>|]', "", name)
    name = re.sub(r"\s+", " ", name).strip()
    return name[:max_len] if name else "untitled_paper"


# -------------------------
# MAIN METADATA SAVE FUNCTION
# -------------------------
def save_metadata_assets(rows: List[Dict[str, Any]], base_dir: str):
    import pandas as pd

    healthcare_rows = [r for r in rows if r.get("healthcare")]

    for row in healthcare_rows:
        specialty = row.get("specialty", "Other")
        specialty_slug = _slugify(specialty)
        specialty_dir = os.path.join(base_dir, specialty_slug)

        metadata_dir = os.path.join(specialty_dir, "Metadata")
        images_root = os.path.join(metadata_dir, "images")
        os.makedirs(images_root, exist_ok=True)

        paper_title = row.get("title", "")
        paper_id = row.get("arxiv_id", "unknown")
        paper_folder_name = safe_folder_name(f"{paper_id}_{paper_title}")
        paper_image_dir = os.path.join(images_root, paper_folder_name)
        os.makedirs(paper_image_dir, exist_ok=True)

        pdf_url = row.get("pdf_url", "")
        if not pdf_url:
            print(f"[Metadata] Skip {row.get('arxiv_id')} -> no pdf_url")
            continue

        raw_csv_path = os.path.join(metadata_dir, "metadata_dataset_links_raw.csv")
        pretty_csv_path = os.path.join(metadata_dir, "metadata_dataset_links.csv")
        useful_csv_path = os.path.join(metadata_dir, "metadata_dataset_links_useful_only.csv")

        if os.path.exists(raw_csv_path):
            df_existing = pd.read_csv(raw_csv_path)
        else:
            df_existing = pd.DataFrame(columns=[
                "arxiv_id",
                "title",
                "specialty",
                "pdf_url",
                "dataset_link",
                "dataset_mentions",
                "published",
                "status",
                "usefulness_label",
                "usefulness_reason",
                "usefulness_score",
                "matched_reference_dataset",
                "matched_reference_sheet",
                "matched_reference_specialty",
                "matched_reference_modality",
                "matched_reference_broad_group",
            ])

        try:
            dataset_info = extract_dataset_info_from_pdf_url(pdf_url)
            dataset_links = dataset_info.get("dataset_links", [])
            dataset_mentions = dataset_info.get("dataset_mentions", [])

            new_rows = []

            if dataset_links:
                for link in dataset_links:
                    usefulness = classify_dataset_usefulness(
                        row=row,
                        dataset_link=link,
                        dataset_mentions=dataset_mentions,
                    )

                    new_rows.append({
                        "arxiv_id": row.get("arxiv_id", ""),
                        "title": paper_title,
                        "specialty": specialty,
                        "pdf_url": pdf_url,
                        "dataset_link": link,
                        "dataset_mentions": " || ".join(dataset_mentions[:5]),
                        "published": row.get("published", ""),
                        "status": "DATASET_LINK_FOUND",
                        "usefulness_label": usefulness["usefulness_label"],
                        "usefulness_reason": usefulness["usefulness_reason"],
                        "usefulness_score": usefulness["usefulness_score"],
                        "matched_reference_dataset": usefulness["matched_reference_dataset"],
                        "matched_reference_sheet": usefulness["matched_reference_sheet"],
                        "matched_reference_specialty": usefulness["matched_reference_specialty"],
                        "matched_reference_modality": usefulness["matched_reference_modality"],
                        "matched_reference_broad_group": usefulness["matched_reference_broad_group"],
                    })
            else:
                usefulness = classify_dataset_usefulness(
                    row=row,
                    dataset_link="",
                    dataset_mentions=dataset_mentions,
                )

                new_rows.append({
                    "arxiv_id": row.get("arxiv_id", ""),
                    "title": paper_title,
                    "specialty": specialty,
                    "pdf_url": pdf_url,
                    "dataset_link": "",
                    "dataset_mentions": " || ".join(dataset_mentions[:5]),
                    "published": row.get("published", ""),
                    "status": "NO_DATASET_LINK_FOUND",
                    "usefulness_label": usefulness["usefulness_label"],
                    "usefulness_reason": usefulness["usefulness_reason"],
                    "usefulness_score": usefulness["usefulness_score"],
                    "matched_reference_dataset": usefulness["matched_reference_dataset"],
                    "matched_reference_sheet": usefulness["matched_reference_sheet"],
                    "matched_reference_specialty": usefulness["matched_reference_specialty"],
                    "matched_reference_modality": usefulness["matched_reference_modality"],
                    "matched_reference_broad_group": usefulness["matched_reference_broad_group"],
                })

            df_new = pd.DataFrame(new_rows)

            for col in [
                "arxiv_id",
                "title",
                "specialty",
                "pdf_url",
                "dataset_link",
                "dataset_mentions",
                "published",
                "status",
                "usefulness_label",
                "usefulness_reason",
                "usefulness_score",
                "matched_reference_dataset",
                "matched_reference_sheet",
                "matched_reference_specialty",
                "matched_reference_modality",
                "matched_reference_broad_group",
            ]:
                if col not in df_existing.columns:
                    df_existing[col] = ""
                if col not in df_new.columns:
                    df_new[col] = ""

                df_existing[col] = df_existing[col].fillna("").astype(str)
                df_new[col] = df_new[col].fillna("").astype(str)

            df_combined = pd.concat([df_existing, df_new], ignore_index=True)

            df_combined.drop_duplicates(
                subset=["arxiv_id", "dataset_link", "status"],
                inplace=True
            )

            df_combined.sort_values(
                by=["arxiv_id", "dataset_link"],
                inplace=True
            )

            df_combined = df_combined.fillna("").reset_index(drop=True)

            # save RAW csv
            df_combined.to_csv(raw_csv_path, index=False)

            # build PRETTY csv from raw
            pretty_rows = []

            for arxiv_id, group in df_combined.groupby("arxiv_id", sort=False):
                group = group.reset_index(drop=True)

                for i, (_, r) in enumerate(group.iterrows()):
                    if i == 0:
                        pretty_rows.append({
                            "arxiv_id": r["arxiv_id"],
                            "title": r["title"],
                            "specialty": r["specialty"],
                            "pdf_url": r["pdf_url"],
                            "dataset_link": r["dataset_link"],
                            "dataset_mentions": r["dataset_mentions"],
                            "published": r["published"],
                            "status": r["status"],
                            "usefulness_label": r["usefulness_label"],
                            "usefulness_reason": r["usefulness_reason"],
                            "usefulness_score": r["usefulness_score"],
                            "matched_reference_dataset": r["matched_reference_dataset"],
                            "matched_reference_sheet": r["matched_reference_sheet"],
                            "matched_reference_specialty": r["matched_reference_specialty"],
                            "matched_reference_modality": r["matched_reference_modality"],
                            "matched_reference_broad_group": r["matched_reference_broad_group"],
                        })
                    else:
                        pretty_rows.append({
                            "arxiv_id": "",
                            "title": "",
                            "specialty": "",
                            "pdf_url": "",
                            "dataset_link": r["dataset_link"],
                            "dataset_mentions": "",
                            "published": "",
                            "status": "",
                            "usefulness_label": r["usefulness_label"],
                            "usefulness_reason": r["usefulness_reason"],
                            "usefulness_score": r["usefulness_score"],
                            "matched_reference_dataset": r["matched_reference_dataset"],
                            "matched_reference_sheet": r["matched_reference_sheet"],
                            "matched_reference_specialty": r["matched_reference_specialty"],
                            "matched_reference_modality": r["matched_reference_modality"],
                            "matched_reference_broad_group": r["matched_reference_broad_group"],
                        })

            df_pretty = pd.DataFrame(pretty_rows).fillna("")
            df_pretty.to_csv(pretty_csv_path, index=False)

            # save USEFUL ONLY csv
            df_useful = df_combined[
                df_combined["usefulness_label"].isin(["KEEP_REFERENCE_MATCH", "REVIEW_TOPIC_MATCH"])
            ].copy()
            df_useful.to_csv(useful_csv_path, index=False)

            print(f"[Metadata RAW] Updated -> {raw_csv_path} (rows={len(df_combined)})")
            print(f"[Metadata CSV] Updated -> {pretty_csv_path} (rows={len(df_pretty)})")
            print(f"[Metadata Useful] Updated -> {useful_csv_path} (rows={len(df_useful)})")

        except Exception as e:
            print(f"[Metadata] Dataset extraction failed for {row.get('arxiv_id')}: {e}")

        # -------------------------
        # IMAGE EXTRACTION
        # -------------------------
        try:
            existing_files = [
                f for f in os.listdir(paper_image_dir)
                if os.path.isfile(os.path.join(paper_image_dir, f))
            ]

            if existing_files:
                print(f"[Images] Skip {row.get('arxiv_id')} -> images already exist")
            else:
                saved = extract_images_from_pdf_url(
                    pdf_url=pdf_url,
                    output_dir=paper_image_dir,
                )
                print(f"[Images] Saved {len(saved)} images for {row.get('arxiv_id')}")

        except Exception as e:
            print(f"[Images] Extraction failed for {row.get('arxiv_id')}: {e}")

In [ ]:
import os
import re
import unicodedata
import pandas as pd
from difflib import SequenceMatcher
from urllib.parse import urlparse, unquote

REFERENCE_XLSX_PATH = "Copy of Proprietary Training Data Summary - Running Total v2.xlsx"
REFERENCE_SHEETS = ["Data List", "Data List (cornell)"]


def normalize_text(s: str) -> str:
    s = str(s or "")
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode("ascii")
    s = s.lower().strip()
    s = s.replace(".zip", " ").replace(".tar.gz", " ").replace(".tgz", " ").replace(".csv", " ")
    s = re.sub(r"[_/\-]+", " ", s)
    s = re.sub(r"[^a-z0-9\s]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


STOPWORDS = {
    "dataset", "data", "images", "image", "study", "paper", "public", "available",
    "the", "and", "for", "with", "from", "using", "used", "zip"
}


def text_to_tokens(s: str) -> set:
    s = normalize_text(s)
    return {tok for tok in s.split() if len(tok) > 2 and tok not in STOPWORDS}


def overlap_score(a_tokens: set, b_tokens: set) -> float:
    if not a_tokens or not b_tokens:
        return 0.0
    inter = len(a_tokens & b_tokens)
    return inter / max(1, min(len(a_tokens), len(b_tokens)))


def similarity_ratio(a: str, b: str) -> float:
    a = normalize_text(a)
    b = normalize_text(b)
    if not a or not b:
        return 0.0
    return SequenceMatcher(None, a, b).ratio()


def get_url_tail(url: str) -> str:
    try:
        path = unquote(urlparse(url).path)
        tail = os.path.basename(path)
        return normalize_text(tail)
    except Exception:
        return ""


def load_reference_catalog(xlsx_path: str = REFERENCE_XLSX_PATH) -> pd.DataFrame:
    frames = []

    for sheet_name in REFERENCE_SHEETS:
        df = pd.read_excel(xlsx_path, sheet_name=sheet_name)

        needed_cols = [
            "File Name",
            "Broad Group",
            "Specialty",
            "Modality in Bucket",
            "Nimblemind/Customer/Non-Commercial",
        ]

        for col in needed_cols:
            if col not in df.columns:
                df[col] = ""

        df = df[needed_cols].copy()
        df["source_sheet"] = sheet_name
        df = df[df["File Name"].notna()].copy()
        df["File Name"] = df["File Name"].astype(str).str.strip()
        df = df[df["File Name"] != ""]

        df["ref_name_norm"] = df["File Name"].apply(normalize_text)
        df["ref_tokens"] = (
            df["File Name"].fillna("").astype(str) + " " +
            df["Broad Group"].fillna("").astype(str) + " " +
            df["Specialty"].fillna("").astype(str) + " " +
            df["Modality in Bucket"].fillna("").astype(str)
        ).apply(text_to_tokens)

        frames.append(df)

    ref_df = pd.concat(frames, ignore_index=True).drop_duplicates(
        subset=["source_sheet", "File Name"]
    ).reset_index(drop=True)

    return ref_df


REFERENCE_CATALOG = load_reference_catalog()


def classify_dataset_usefulness(
    row: dict,
    dataset_link: str,
    dataset_mentions,
    ref_df: pd.DataFrame = REFERENCE_CATALOG
) -> dict:
    dataset_mentions = dataset_mentions or []
    mention_text = " ".join(dataset_mentions[:5]) if isinstance(dataset_mentions, list) else str(dataset_mentions)

    title = row.get("title", "")
    abstract = row.get("abstract", "")
    specialty = row.get("specialty", "")
    pdf_url = row.get("pdf_url", "")

    candidate_text = " ".join([
        title,
        abstract,
        specialty,
        dataset_link or "",
        mention_text,
        pdf_url,
    ])

    candidate_norm = normalize_text(candidate_text)
    candidate_tokens = text_to_tokens(candidate_text)

    link_tail = get_url_tail(dataset_link or "")
    paper_specialty_tokens = text_to_tokens(specialty)

    best = None
    best_score = -1

    for _, ref in ref_df.iterrows():
        ref_name = ref["File Name"]
        ref_name_norm = ref["ref_name_norm"]
        ref_tokens = ref["ref_tokens"]

        name_score = max(
            similarity_ratio(link_tail, ref_name_norm),
            similarity_ratio(candidate_norm, ref_name_norm) if len(ref_name_norm) < 40 else 0.0
        )

        token_score = overlap_score(candidate_tokens, ref_tokens)

        ref_specialty_tokens = text_to_tokens(ref.get("Specialty", ""))
        ref_modality_tokens = text_to_tokens(ref.get("Modality in Bucket", ""))
        ref_broad_tokens = text_to_tokens(ref.get("Broad Group", ""))

        specialty_score = overlap_score(paper_specialty_tokens, ref_specialty_tokens)
        modality_score = overlap_score(candidate_tokens, ref_modality_tokens)
        broad_score = overlap_score(candidate_tokens, ref_broad_tokens)

        total_score = (
            0.50 * name_score +
            0.25 * token_score +
            0.15 * specialty_score +
            0.05 * modality_score +
            0.05 * broad_score
        )

        if total_score > best_score:
            best_score = total_score
            best = {
                "matched_reference_dataset": ref_name,
                "matched_reference_sheet": ref.get("source_sheet", ""),
                "matched_reference_specialty": ref.get("Specialty", ""),
                "matched_reference_modality": ref.get("Modality in Bucket", ""),
                "matched_reference_broad_group": ref.get("Broad Group", ""),
                "name_score": round(name_score, 4),
                "token_score": round(token_score, 4),
                "specialty_score": round(specialty_score, 4),
                "modality_score": round(modality_score, 4),
                "broad_score": round(broad_score, 4),
                "usefulness_score": round(total_score, 4),
            }

    if not best:
        return {
            "usefulness_label": "DROP_NOT_USEFUL",
            "usefulness_reason": "No reference match found.",
            "usefulness_score": 0.0,
            "matched_reference_dataset": "",
            "matched_reference_sheet": "",
            "matched_reference_specialty": "",
            "matched_reference_modality": "",
            "matched_reference_broad_group": "",
            "name_score": 0.0,
            "token_score": 0.0,
            "specialty_score": 0.0,
            "modality_score": 0.0,
            "broad_score": 0.0,
        }

    # ===== Decision thresholds (UPDATED WITH SOFT RULE) =====
    if best["name_score"] >= 0.92 or best["usefulness_score"] >= 0.78:
        label = "KEEP_REFERENCE_MATCH"
        reason = "Strong match to reference dataset/topic catalog."

    elif best["specialty_score"] >= 0.9 and best["modality_score"] >= 0.9:
        label = "REVIEW_TOPIC_MATCH"
        reason = "Strong specialty + modality match, but weaker dataset name match."

    elif best["usefulness_score"] >= 0.52:
        label = "REVIEW_TOPIC_MATCH"
        reason = "Moderate topic match to reference catalog; review recommended."

    else:
        label = "DROP_NOT_USEFUL"
        reason = "Weak match to reference datasets/topics."

    return {
        **best,
        "usefulness_label": label,
        "usefulness_reason": reason,
    }

# Testing - Dataset Classifier

In [ ]:
print("Reference catalog rows:", len(REFERENCE_CATALOG))
print(REFERENCE_CATALOG[[
    "File Name",
    "Broad Group",
    "Specialty",
    "Modality in Bucket",
    "source_sheet"
]].head(10))

Reference catalog rows: 178
                      File Name Broad Group         Specialty  \
0                 PolypsSet.zip   2D Images  Gastroenterology   
1     Breast_Histopathology.zip   2D Images         Pathology   
2                      SNOW.zip   2D Images         Pathology   
3           Skin_DNA_Damage.zip   2D Images         Pathology   
4  DeepLesion_DeepLesion_01.zip    CT scans         Radiology   
5         colorectal_cancer.zip   2D Images         Pathology   
6                      CXR8.zip   2D Images         Radiology   
7              PatchGastric.zip   2D Images         Pathology   
8                  Path_VQA.zip   2D Images         Pathology   
9                    PMC_OA.zip   2D Images         Radiology   

  Modality in Bucket source_sheet  
0            Imaging    Data List  
1            Imaging    Data List  
2            Imaging    Data List  
3            Imaging    Data List  
4            Imaging    Data List  
5            Imaging    Data List  
6   

In [ ]:
test_row = {
    "arxiv_id": "test-001",
    "title": "A study using public oncology imaging datasets for tumor classification",
    "abstract": "We used a public cancer imaging dataset and benchmark data for model development.",
    "specialty": "Oncology",
    "pdf_url": "https://example.com/test.pdf",
}

test_dataset_link = "https://openneuro.org/datasets/ds003097/versions/1.2.1"
test_dataset_mentions = [
    "We evaluated the method on a public dataset for tumor classification.",
    "The dataset is openly available and commonly used for benchmarking."
]

result = classify_dataset_usefulness(
    row=test_row,
    dataset_link=test_dataset_link,
    dataset_mentions=test_dataset_mentions,
)

print(result)

{'matched_reference_dataset': 'LIDC-IDRI_CT.zip', 'matched_reference_sheet': 'Data List', 'matched_reference_specialty': 'Pulmonology / Oncology', 'matched_reference_modality': 'Imaging', 'matched_reference_broad_group': '3D Imaging', 'name_score': 0.2353, 'token_score': 0.4, 'specialty_score': 1.0, 'modality_score': 1.0, 'broad_score': 1.0, 'usefulness_score': 0.4676, 'usefulness_label': 'REVIEW_TOPIC_MATCH', 'usefulness_reason': 'Strong specialty + modality match, but weaker dataset name match.'}


In [ ]:
for k, v in result.items():
    print(f"{k}: {v}")

matched_reference_dataset: LIDC-IDRI_CT.zip
matched_reference_sheet: Data List
matched_reference_specialty: Pulmonology / Oncology
matched_reference_modality: Imaging
matched_reference_broad_group: 3D Imaging
name_score: 0.2353
token_score: 0.4
specialty_score: 1.0
modality_score: 1.0
broad_score: 1.0
usefulness_score: 0.4676
usefulness_label: REVIEW_TOPIC_MATCH
usefulness_reason: Strong specialty + modality match, but weaker dataset name match.


In [ ]:
test_pdf_url = "https://arxiv.org/pdf/2301.00785v5"

dataset_info = extract_dataset_info_from_pdf_url(test_pdf_url)

test_row_real = {
    "arxiv_id": "2301.03459v1",
    "title": "test paper",
    "abstract": "",
    "specialty": "Oncology",
    "pdf_url": test_pdf_url,
}

if dataset_info["dataset_links"]:
    for link in dataset_info["dataset_links"]:
        result = classify_dataset_usefulness(
            row=test_row_real,
            dataset_link=link,
            dataset_mentions=dataset_info["dataset_mentions"],
        )
        print("\nLINK:", link)
        for k, v in result.items():
            print(f"{k}: {v}")
else:
    result = classify_dataset_usefulness(
        row=test_row_real,
        dataset_link="",
        dataset_mentions=dataset_info["dataset_mentions"],
    )
    print("\nNO DATASET LINK FOUND")
    for k, v in result.items():
        print(f"{k}: {v}")


LINK: https://github.com/ljwztc/CLIP-Driven-Universal-Model
matched_reference_dataset: LIDC-IDRI_CT.zip
matched_reference_sheet: Data List
matched_reference_specialty: Pulmonology / Oncology
matched_reference_modality: Imaging
matched_reference_broad_group: 3D Imaging
name_score: 0.359
token_score: 0.4
specialty_score: 1.0
modality_score: 1.0
broad_score: 1.0
usefulness_score: 0.5295
usefulness_label: REVIEW_TOPIC_MATCH
usefulness_reason: Strong specialty + modality match, but weaker dataset name match.

LINK: https://github.com/openai/CLIP
matched_reference_dataset: LIDC-IDRI_CT.zip
matched_reference_sheet: Data List
matched_reference_specialty: Pulmonology / Oncology
matched_reference_modality: Imaging
matched_reference_broad_group: 3D Imaging
name_score: 0.25
token_score: 0.4
specialty_score: 1.0
modality_score: 1.0
broad_score: 1.0
usefulness_score: 0.475
usefulness_label: REVIEW_TOPIC_MATCH
usefulness_reason: Strong specialty + modality match, but weaker dataset name match.


#4. Agent

In [ ]:
def run_pipeline(
    start_date: str,
    end_date: str,
    categories=None,
    max_results_total: int = 200,
):
    print("=" * 60)
    print("Agent started")
    print(f"Date range: {start_date} to {end_date}")

    # Step 1: Fetch papers
    papers = fetch_papers_by_date_range(
        start_date=start_date,
        end_date=end_date,
        categories=categories or DEFAULT_CATEGORIES,
        max_results_total=max_results_total,
    )

    print(f"Fetched {len(papers)} papers")

    # Optional: save fetched papers
    fetched_path = os.path.join(FETCHED_DIR, f"{start_date}_{end_date}_fetched.csv")
    save_rows_to_csv(papers, fetched_path)

    # Step 2: Classify
    rows, summary = classify_batch(papers)

    print("Classification summary:")
    for k, v in summary.items():
        print(f"  {k}: {v}")

    # Step 3: Extract full text + abstract
    save_full_texts_by_specialty(
        rows,
        base_dir=CLASSIFIED_DIR,
    )

    # Step 4: Save metadata sheet + extracted images
    save_metadata_assets(
        rows,
        base_dir=CLASSIFIED_DIR,
    )

    # Step 5: Save classified CSV outputs
    save_specialty_outputs(
        rows,
        base_dir=CLASSIFIED_DIR,
        start_date=start_date,
        end_date=end_date,
    )

    # Step 6: Save summary
    summary_path = os.path.join(CLASSIFIED_DIR, f"{start_date}_{end_date}_summary.csv")
    save_run_summary(summary, summary_path)

    print("Pipeline complete")
    print("=" * 60)

    return papers, rows, summary

In [ ]:
papers, rows, summary = run_pipeline(
    start_date="2023-01-01",
    end_date="2023-01-31",
    max_results_total=100,
)

Agent started
Date range: 2023-01-01 to 2023-01-31


NameError: name 'fetch_papers_by_date_range' is not defined